In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold , cross_val_score, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

In [18]:
df=pd.read_csv("F:\MACHINE LEARNING\SOMETHING\Cases\Vehicle Silhouettes\Vehicle.csv")
X,y=df.drop('Class',axis=1),df.Class
le=LabelEncoder()
y=le.fit_transform(y)
kfold = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 26)
gbm = GradientBoostingClassifier(random_state = 26)

Accuracy Score

In [5]:
results = cross_val_score(gbm, X, y, cv = kfold)
results

array([0.77058824, 0.76923077, 0.78698225, 0.78106509, 0.75147929])

In [4]:
results = cross_val_score(gbm, X, y, cv = kfold, scoring= 'neg_log_loss')
results

array([-0.46420906, -0.47148921, -0.43516682, -0.47498426, -0.55013841])

In [ ]:
rates=np.linspace(.01,.9,10)
depth=[None,2,3,4]
n_est=[25,50,100]
scores=[]
for r in tqdm(rates):
    for n in n_est:
        for d in depth:
            gbm=GradientBoostingClassifier(random_state=26,max_depth=d,n_estimators=n,learning_rate=r)
            kfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
            results=cross_val_score(gbm,X,y,cv=kfold,scoring='neg_log_loss')
            scores.append([r, n, d, results.mean()])
df_scores = pd.DataFrame(scores, columns= ['rates','n_estimator','depth','log_loss'])


In [6]:
df_scores.sort_values(by='log_loss')

NameError: name 'df_scores' is not defined

In [7]:
solver=['lbfgs','newton-cholesky','newton-cg']
scores1=[]
for s in tqdm(solver):
    lr=LogisticRegression(solver=s)
    kfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
    results=cross_val_score(lr,X,y,cv=kfold,scoring='neg_log_loss')
    scores1.append([s, results.mean()])
df_scores1 = pd.DataFrame(scores1, columns= ['SOLVER','log_loss'])
df_scores1.sort_values(by='log_loss')
    

100%|██████████| 3/3 [00:03<00:00,  1.25s/it]


,SOLVER,log_loss
0,lbfgs,-0.679946
1,newton-cholesky,-0.438304
2,newton-cg,-0.416661


In [8]:
solver=['lbfgs','newton-cholesky','newton-cg']
cs = np.linspace(0.01,3,5)
scores2=[]
for s in tqdm(solver):
    for c in cs:
        lr=LogisticRegression(solver=s, C = c )
        kfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
        results=cross_val_score(lr,X,y,cv=kfold,scoring='neg_log_loss')
        scores2.append([s,c,results.mean()])
df_scores2 = pd.DataFrame(scores2, columns= ['SOLVER', 'C' ,'log_loss'])
df_scores2.sort_values(by='log_loss',ascending=False)

100%|██████████| 3/3 [00:20<00:00,  6.84s/it]


,SOLVER,C,log_loss
11,newton-cg,0.7575,-0.416719
12,newton-cg,1.5050,-0.425281
13,newton-cg,2.2525,-0.427607
14,newton-cg,3.0000,-0.431577
6,newton-cholesky,0.7575,-0.432451
7,newton-cholesky,1.5050,-0.447914
8,newton-cholesky,2.2525,-0.458194
9,newton-cholesky,3.0000,-0.465749
5,newton-cholesky,0.0100,-0.470541
10,newton-cg,0.0100,-0.472216


In [9]:
lr=LogisticRegression()
params = {'solver': ['lbfgs','newton-cholesky','newton-cg'],
          'C': np.linspace(0.01,3,5)}
gcv = GridSearchCV(lr, param_grid = params, cv = kfold, scoring = 'neg_log_loss')
gcv.fit(X,y)
print('Best parameter: ',gcv.best_params_)
print('Best score: ',gcv.best_score_)

Best parameter:  {'C': np.float64(0.7575000000000001), 'solver': 'newton-cg'}
Best score:  -0.4167192131442695


In [21]:
concrete = pd.read_csv("F:\MACHINE LEARNING\SOMETHING\Cases\Concrete_Strength\Concrete_Data.csv")
X,y=concrete.drop('Strength',axis=1),concrete.Strength
concrete

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30
...,...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28,44.28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28,31.18
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28,23.70
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28,32.77


In [22]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

gbm = XGBRegressor(random_state=26)
params = {'learning_rate':np.linspace(0.01, 0.9, 10),
          'max_depth': [None, 2,3,4],
          'n_estimators': [25,50,100]}
kfold=KFold(n_splits=5,shuffle=True,random_state=26)
gcv = GridSearchCV(gbm, param_grid = params, cv = kfold, scoring = 'r2')
gcv.fit(X,y)
print('Best parameter: ',gcv.best_params_)
print('Best score: ',gcv.best_score_)

Best parameter:  {'learning_rate': np.float64(0.6033333333333333), 'max_depth': 3, 'n_estimators': 100}
Best score:  0.935401012517735


In [23]:
bm = XGBRegressor(**gcv.best_params_ , random_state=26)
bm.fit(X,y)
tst = pd.read_csv(r"F:\MACHINE LEARNING\SOMETHING\Cases\Concrete_Strength\testConcrete.csv")
bm.predict(tst)

array([68.533485, 44.278156, 43.30317 , 36.11119 , 50.87477 , 13.594891,
       46.644585, 65.02611 , 53.04127 , 56.998302, 53.745834, 47.816093,
       45.142815, 43.668945], dtype=float32)

In [20]:
tst = pd.read_csv(r"F:\MACHINE LEARNING\SOMETHING\Cases\Concrete_Strength\testConcrete.csv")
tst

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,495,120,0,155,5,866,884,75
1,262,129,0,271,2,808,787,174
2,201,48,1,215,5,807,839,113
3,329,141,0,286,1,881,823,229
4,354,14,0,129,2,839,847,210
5,150,23,23,114,4,883,638,36
6,480,64,0,292,3,896,776,180
7,393,49,82,132,1,887,830,271
8,284,63,1,138,1,804,725,44
9,206,38,0,103,2,818,719,191
